In [1]:
from app.data.etl import GetData
from app.data.feature_engineering import FeatureEngineering

In [2]:
get_data = GetData(file_path='app/data/raw_data/BD_ordenes.xlsx')
df = get_data.read_data()

In [3]:
df.head()

,respuesta,consumo_criticado,servicio,categoria,nivel_tension,estrato,localidad,funcion_analisis,calificacion,obs_lectura,periodicidad
0,1,0.0,701-ENERGÍA MDO REGULADO,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5035-BAJO ENERGIA (<-50%),30-VARIACION NIVEL DE UTILIZACIÓN,1
1,1,420.0,701-ENERGÍA MDO REGULADO,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5080-MUY ALTO (>500%),30-VARIACION NIVEL DE UTILIZACIÓN,1
2,1,99999.0,101-AGUA POTABLE,1-RESIDENCIAL,NaN,1.0,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),34-LECTURA MENOR,1
3,1,881.0,701-ENERGÍA MDO REGULADO,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),35-NO HAY JUSTIFICACION,1
4,1,99999.0,701-ENERGÍA MDO REGULADO,1-RESIDENCIAL,220.0,4.0,5001-MEDELLÍN,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),34-LECTURA MENOR,1


# Feature Engineering

In [4]:
feature_engineering = FeatureEngineering(df)
df_fe, target_variable, numeric_features, categorical_features = feature_engineering.prepare_features_only_energy()

In [5]:
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target variable:", target_variable)

Numeric features: ['consumo_criticado', 'nivel_tension', 'estrato', 'periodicidad']
Categorical features: ['categoria', 'localidad', 'funcion_analisis', 'calificacion', 'obs_lectura', 'tipo_servicio']
Target variable: respuesta


In [6]:
df_fe.head()

,respuesta,consumo_criticado,categoria,nivel_tension,estrato,localidad,funcion_analisis,calificacion,obs_lectura,periodicidad,tipo_servicio
0,0,0.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5035-BAJO ENERGIA (<-50%),30-VARIACION NIVEL DE UTILIZACIÓN,1,Energia
1,0,420.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5080-MUY ALTO (>500%),30-VARIACION NIVEL DE UTILIZACIÓN,1,Energia
3,0,881.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),35-NO HAY JUSTIFICACION,1,Energia
4,0,99999.0,1-RESIDENCIAL,220.0,4.0,5001-MEDELLÍN,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),34-LECTURA MENOR,1,Energia
7,0,260.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5035-BAJO ENERGIA (<-50%),30-VARIACION NIVEL DE UTILIZACIÓN,1,Energia


In [7]:
test_size = 0.2

# Modelo con MLFlow + Optuna

Antes de ejecutar, subir cliente de MLFlow con el comando.

* mlflow server --backend-store-uri sqlite:///consumos.db

In [8]:
from sklearn.ensemble import RandomForestClassifier
from app.train.train import TrainModel
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Define RandomForest parameter distributions
param_distributions = {
    'n_estimators': ('int', 50, 200),
    'max_depth': ('int', 5, 30),
    'min_samples_split': ('int', 2, 10),
    'min_samples_leaf': ('int', 1, 5),
    'max_features': ('categorical', ['sqrt', 'log2', None])
}

trainer = TrainModel(
    df_fe,
    numeric_features = numeric_features,
    categorical_features = categorical_features,
    target_column = target_variable,
    model_class = RandomForestClassifier,
    test_size = test_size,
    model_params = {'random_state': 42, 'n_jobs': -1},
    param_distributions = param_distributions,
    n_trials = 20,
    optimization_metric = 'f1',
    mlflow_setup = mlflow,
    mlflow_registered_model_name = "RF_Desviacion_Consumos"
)

best_pipeline, run_id, study = trainer.train()

[I 2025-10-18 17:45:18,757] A new study created in memory with name: optuna_RandomForestClassifier
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/src/app/train/train.py:186: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
INFO:app.train.train:Starting Optuna optimization with 20 trials...
INFO:app.train.train:Optimizing for: f1
INFO:app.train.train:Model type: RandomForestClassifier
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:18,969] Trial 0 finished with value: 0.8015627747900393 and parameters: {'n_estimators': 108, 'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'log2'}. Be

🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/1/runs/ebf7d8ebe4e745efb2b831b41fff4e67
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run 1 at: http://127.0.0.1:5000/#/experiments/1/runs/7aafd43733a24fd598add8995c7548a6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:20,737] Trial 2 finished with value: 0.822801590286785 and parameters: {'n_estimators': 194, 'max_depth': 14, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 2 with value: 0.822801590286785.


🏃 View run 2 at: http://127.0.0.1:5000/#/experiments/1/runs/80b365d36e034f79a7ccfb35f8ed6c6a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:21,723] Trial 3 finished with value: 0.8257440833663792 and parameters: {'n_estimators': 166, 'max_depth': 13, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 3 with value: 0.8257440833663792.
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:21,929] Trial 4 finished with value: 0.6864372700808955 and parameters: {'n_estimators': 134, 'max_depth': 11, 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_

...

🏃 View run 17 at: http://127.0.0.1:5000/#/experiments/1/runs/e8e4b0c1b4a24d159839d76392f170ea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:28,649] Trial 18 finished with value: 0.8061424545617839 and parameters: {'n_estimators': 124, 'max_depth': 20, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 11 with value: 0.8272236262465439.
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:28,826] Trial 19 finished with value: 0.753520778667868 and parameters: {'n_estimators': 85, 'max_depth': 14, 'min_samples_split': 5, 'min_samples_leaf': 1, 'm

🏃 View run 18 at: http://127.0.0.1:5000/#/experiments/1/runs/8d107ae1713a4ebcb0f9d7c626559c28
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run 19 at: http://127.0.0.1:5000/#/experiments/1/runs/56b441187c6c4026acb7d8fcaf028c25
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/do

🏃 View run best_model_RandomForestClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/d669e1d726a24d538bdede523c8c0096
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [10]:
print(f"Best MLflow run ID: {run_id}")  
print(f"Best hyperparameters: {study.best_params}")

Best MLflow run ID: d669e1d726a24d538bdede523c8c0096
Best hyperparameters: {'n_estimators': 146, 'max_depth': 27, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


In [11]:
from xgboost import XGBClassifier

In [12]:
# Define XGBoost parameter distributions
param_distributions_xgb = {
    'n_estimators': ('int', 100, 1000),
    'max_depth': ('int', 3, 10),
    'learning_rate': ('float', 0.01, 0.3, True), # Log scale es bueno para learning rate
    'subsample': ('float', 0.5, 1.0),
    'colsample_bytree': ('float', 0.5, 1.0),
    'gamma': ('float', 0, 0.5),
    'reg_alpha': ('float', 1e-5, 1.0, True), # L1 regularization
    'reg_lambda': ('float', 1e-5, 1.0, True)  # L2 regularization
}

fixed_model_params_xgb = {
    'random_state': 42, 
    'n_jobs': -1,
    'objective': 'binary:logistic', # clasificación binaria
    'eval_metric': 'logloss' 
}

trainer_xgb = TrainModel(
    df_fe,
    numeric_features = numeric_features,
    categorical_features = categorical_features,
    target_column = target_variable,
    model_class = XGBClassifier,  # <-- Cambio clave
    test_size = test_size,
    model_params = fixed_model_params_xgb, # <-- Cambio clave
    param_distributions = param_distributions_xgb, # <-- Cambio clave
    n_trials = 20, # Puedes ajustar esto
    optimization_metric = 'roc_auc', # 'roc_auc' es excelente para XGBoost
    mlflow_setup = mlflow,
    mlflow_registered_model_name = "XGB_Desviacion_Consumos"  # Nombre para el registro del modelo
)

best_pipeline_xgb, run_id_xgb, study_xgb = trainer_xgb.train()

print("\n--- Entrenamiento de XGBoost completado ---")
print(f"Mejor Pipeline (XGB): {best_pipeline_xgb}")
print(f"MLflow Run ID (XGB): {run_id_xgb}")

[I 2025-10-18 17:45:45,022] A new study created in memory with name: optuna_XGBClassifier
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/src/app/train/train.py:186: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
INFO:app.train.train:Starting Optuna optimization with 20 trials...
INFO:app.train.train:Optimizing for: roc_auc
INFO:app.train.train:Model type: XGBClassifier
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:45,859] Trial 0 finished with value: 0.9002075794972563 and parameters: {'n_estimators': 602, 'max_depth': 5, 'learning_rate': 0.2087938899780242, 'subsample': 0.8413411566717752, 'colsample_bytree'

🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/2/runs/fc516f5862694d8a96ceb7d4e2602901
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:46,238] Trial 1 finished with value: 0.8961358662225524 and parameters: {'n_estimators': 301, 'max_depth': 4, 'learning_rate': 0.12800419530381202, 'subsample': 0.9976741503607557, 'colsample_bytree': 0.9330804780992166, 'gamma': 0.4587475359910426, 'reg_alpha': 1.620311061469145e-05, 'reg_lambda': 0.0007631871901243023}. Best is trial 0 with value: 0.9002075794972563.


🏃 View run 1 at: http://127.0.0.1:5000/#/experiments/2/runs/ddd7f574b693479a96873edd0aa1187d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:45:46,710] Trial 2 finished with value: 0.9001969958384255 and parameters: {'n_estimators': 312, 'max_depth': 5, 'learning_rate': 0.17913906943342622, 'subsample': 0.744548322549925, 'colsample_bytree': 0.9203994835965976, 'gamma': 0.24705913644492555, 'reg_alpha': 4.1357917047279264e-05, 'reg_lambda': 1.0666834700708342e-05}. Best is trial 0 with value: 0.9002075794972563.


...

🏃 View run 18 at: http://127.0.0.1:5000/#/experiments/2/runs/b5d395a680424de3965ca2551f2d1cb7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:46:07,820] Trial 19 finished with value: 0.902169749906145 and parameters: {'n_estimators': 846, 'max_depth': 9, 'learning_rate': 0.03223246322412124, 'subsample': 0.7973421081007609, 'colsample_bytree': 0.5746172188074012, 'gamma': 0.1941063270159612, 'reg_alpha': 0.07321295967907232, 'reg_lambda': 0.16937355503849716}. Best is trial 17 with value: 0.9024682889618427.
INFO:app.train.train:Optimization complete!
INFO:app.train.train:Best roc_auc: 0.9025
INFO:app.train.train:Best parameters: {'n_estimators': 756, 'max_depth': 9, 'learning_rate': 0.028212674990426, 'subsample': 0.7337306614305988, 'colsample_bytree': 0.6931404846836668, 'gamma': 0.19801849988793252, 'reg_alpha': 0.07494076811

🏃 View run 19 at: http://127.0.0.1:5000/#/experiments/2/runs/51fc7be48322481b8c5e8a1c1678334b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/do

🏃 View run best_model_XGBClassifier at: http://127.0.0.1:5000/#/experiments/2/runs/09c9684c0e384d418e62bf214404b2d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- Entrenamiento de XGBoost completado ---
Mejor Pipeline (XGB): Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['consumo_criticado',
                                                   'nivel_tension', 'estrato',
                                                   'periodicidad']),
                                                 ('cat',
                                             